# Day 2 · 5교시 · vLLM 서빙 · OpenAI API · 벤치

> **VLM 경량화 2일 과정 · Day 2 (5교시)** · 실습 환경: **RunPod A100 80GB** · 모델: **Qwen3-VL-8B AWQ**

이 교시는 **서버를 터미널에서 한 번 띄우고**(start/stop 반복 없음), 노트북은 **OpenAI 호환 클라이언트**로만 동작합니다. 비교군(원본 8B)은 GPU를 다시 점유해 느리게 재기동하는 대신, **정확도·크기는 Day2-4 오프라인 평가 수치**를 사용합니다(시간 절약).


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


**▸ 실행 결과 읽기.** `HF_TOKEN: 로드됨` 이면 `.env` 인증 성공, 아래에 작업 폴더 경로가 찍힙니다. 이 두 줄이 보이면 공통 헤더 준비 완료입니다.

## 1. 서빙용 venv 생성

vLLM은 추론 엔진이라 **무거운 CUDA 빌드·의존성**을 끌고 옵니다. 학습/양자화에 쓰던 환경과 섞이면 버전 충돌이 나기 쉬워서, **서빙 전용 가상환경 `.venv-serve`** 에 따로 설치합니다(격리).

- **`uv`**: 매우 빠른 파이썬 패키지 매니저(여기선 venv 생성 + 설치에 사용).
- **서버**(vLLM)는 `.venv-serve`에서, **클라이언트**(openai/requests)는 지금 이 **노트북 커널**에서 돕니다 — 둘은 분리돼 있습니다.
- 설치에 **수 분** 걸릴 수 있습니다(vLLM + torch 다운로드). `$ …` 줄은 실행된 명령이고, 마지막 `설치 완료`가 뜨면 성공입니다.

> 💡 이 pod는 **CUDA 13 드라이버**라 PyPI 기본 vLLM(cu130 빌드)이 그대로 동작합니다. (드라이버가 12.x인 pod라면 cu129 휠을 따로 지정해야 합니다 — 같은 코스, 다른 환경일 때의 함정.)

In [2]:
# ── 서빙 venv + vLLM 설치 ────────────────────────────────────
import subprocess, sys

WORKDIR = VLM_DIR   # 상단 공통 헤더의 작업 폴더 변수 사용
SERVE_VENV = f'{WORKDIR}/.venv-serve'
SERVE_VLLM = f'{SERVE_VENV}/bin/vllm'
SERVE_PY   = f'{SERVE_VENV}/bin/python'

def run(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-1200:])
    if r.returncode != 0: print('STDERR:', r.stderr[-1200:])
    return r.returncode

# uv 는 OS PATH(Day2-2에서 설치)에 있으므로 커널과 무관하게 동작
run(f'uv venv --python 3.12 {SERVE_VENV}')
# 최신 vLLM 설치 (이 pod는 CUDA 13 드라이버 → PyPI 기본 cu130 빌드가 그대로 동작):
#   - 최신 vLLM은 transformers v5 포함 → Qwen3-VL v5 토크나이저를 그대로 읽음.
#   - CUDA 13 드라이버라 cu130(기본) OK. (12.x 드라이버 pod라면 cu129 릴리스 휠을 직접 지정해야 함)
# ninja: vLLM 0.23이 샘플링에 쓰는 FlashInfer가 일부 커널을 런타임 JIT 컴파일 → ninja 빌드도구 필요
run(f'uv pip install --python {SERVE_PY} -U vllm ninja ipykernel ipywidgets')
# datasets는 최신(hub 1.x) — transformers v5와 일관이라 핀 불필요. pyarrow16+ 새 extension API 사용.
run(f'uv pip install --python {SERVE_PY} -U datasets')
# torch의 cusparse가 요구하는 nvJitLink 심볼 처방(유지)
run(f'uv pip install --python {SERVE_PY} -U nvidia-nvjitlink-cu12')
# 클라이언트(노트북 커널)용
run(f'uv pip install --python {sys.executable} -U openai requests datasets')
print('설치 완료')

$ uv venv --python 3.12 /workspace/VLM_FT_2/.venv-serve
$ uv pip install --python /workspace/VLM_FT_2/.venv-serve/bin/python -U vllm ninja ipykernel ipywidgets
$ uv pip install --python /workspace/VLM_FT_2/.venv-serve/bin/python -U datasets
$ uv pip install --python /workspace/VLM_FT_2/.venv-serve/bin/python -U nvidia-nvjitlink-cu12
$ uv pip install --python /usr/bin/python -U openai requests datasets
설치 완료


**▸ 실행 결과 읽기.** `$`로 시작하는 줄은 내부적으로 실행한 설치 명령이고, 각 명령의 핵심 로그 끝부분만 보여줍니다. 마지막에 **`설치 완료`** 가 보이면 서빙 venv 준비가 끝난 것입니다. 클라이언트 패키지(`openai` 등)는 **이 셀을 실행한 커널**에 설치되므로, 아래 [3.]~[6.]의 클라이언트 셀도 **같은 커널**에서 이어 실행하세요.

## 2. vLLM 서버 띄우기 — 터미널

`vllm serve` 는 **종료되지 않고 계속 떠 있는 프로세스**라 노트북 셀에 넣으면 셀이 끝나지 않습니다 → 반드시 **RunPod 터미널 탭**에서 실행합니다.

명령에 붙는 옵션의 뜻:
- **`--max-model-len 4096`**: 한 요청이 다룰 수 있는 최대 토큰 수(입력+출력). 작을수록 KV 캐시가 덜 필요합니다.
- **`--gpu-memory-utilization 0.9`**: GPU 메모리의 90%를 미리 잡아 **KV 캐시(동시 요청용 좌석)** 로 씁니다 → 처리량↑.
- **`--enforce-eager`**: CUDA 그래프 캡처를 건너뛰어 **기동을 수 분 단축**(추론은 약간 느려짐 — 교육/데모엔 이득).
- **`VLLM_USE_FLASHINFER_SAMPLER=0`**: 샘플러의 런타임 JIT 컴파일을 꺼 기동을 더 빠르고 단순하게 만듭니다.

아래 셀이 **실제 경로가 채워진 명령**을 출력합니다. 그대로 복사해 터미널에 붙여넣으세요.

In [4]:
# ── 서버 기동은 '터미널'에서 (노트북 아님!) ────────────────────────
# 아래 출력된 명령을 RunPod 터미널 탭에 그대로 붙여넣어 한 번만 띄우세요.

# OpenAI 호환 API 서버 포트
PORT       = 8000

# OpenAI API 클라이언트에서 사용할 모델 이름 (실제 모델명과 무관, 임의 식별자)
MODEL_NAME = 'model'

# Day2-3에서 생성한 Qwen3-VL 8B AWQ 양자화 모델 디렉토리
AWQ_8B_DIR = f'{VLM_DIR}/Qwen3-VL-8B-AWQ'

# Day2-5 [4]에서 설치한 vLLM 실행 파일 경로 (venv 내)
SERVE_VLLM = f'{VLM_DIR}/.venv-serve/bin/vllm'

# vLLM 서버 기동 명령 구성
# - VLLM_USE_FLASHINFER_SAMPLER=0: FlashInfer 샘플러의 런타임 JIT 컴파일을 비활성화
#   → 기동 시간 단축 & ninja 빌드 도구 의존성 제거
# - --served-model-name: OpenAI API 응답에 표시될 모델 이름
# - --max-model-len: 최대 시퀀스 길이(토큰) 제한 (4096 = Qwen3-VL 기본값 근처)
# - --gpu-memory-utilization 0.9: GPU 메모리의 90%를 KV 캐시로 예약 (처리량 최대화)
# - --enforce-eager: CUDA 그래프 캡처 스킵 → 기동 시간 단축 (추론 속도는 약간 저하)
# - --port: OpenAI 호환 서버 리스닝 포트
cmd = (f'VLLM_USE_FLASHINFER_SAMPLER=0 {SERVE_VLLM} serve {AWQ_8B_DIR} \\\n'
       f'  --served-model-name {MODEL_NAME} \\\n'
       f'  --max-model-len 4096 \\\n'
       f'  --gpu-memory-utilization 0.9 \\\n'
       f'  --enforce-eager \\\n'
       f'  --port {PORT}')

print('▶ RunPod 터미널에 붙여넣기:\n')
print(cmd)

▶ RunPod 터미널에 붙여넣기:

VLLM_USE_FLASHINFER_SAMPLER=0 /workspace/VLM_FT_2/.venv-serve/bin/vllm serve /workspace/VLM_FT_2/Qwen3-VL-8B-AWQ \
  --served-model-name model \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.9 \
  --enforce-eager \
  --port 8000


**▸ 실행 결과 읽기.** 출력된 명령을 **복사해 RunPod 터미널에 붙여넣어** 서버를 띄웁니다. 터미널 로그가 *모델 로드 → 가중치 → KV 캐시 측정* 순으로 흐르다가 마지막에 **`Application startup complete.`** 가 뜨면 준비 완료입니다. 그 줄을 본 뒤 노트북의 다음 셀([3.])로 넘어가세요.

## 3. 서버 준비 확인 (readiness)

서버는 떠도 **모델 로드가 끝나야** 요청을 받습니다. 그래서 곧바로 요청하면 연결이 거부될 수 있어, `/v1/models` 엔드포인트가 **200(정상)** 을 줄 때까지 5초 간격으로 **반복 확인(폴링)** 합니다. 준비되면 통과하고, 정해진 시간(기본 600초)을 넘기면 오류를 냅니다.

In [5]:
# ── 서버 준비 확인 ────────────────────────────────────────────
import time, requests

def wait_ready(timeout=600):
    # OpenAI 호환 API 서버의 모델 목록 엔드포인트
    url = f'http://localhost:{PORT}/v1/models'
    t0 = time.time()
    # 최대 timeout(초) 동안 서버 응답 대기
    while time.time() - t0 < timeout:
        try:
            # 서버가 준비되면 /v1/models에서 200 상태 반환
            if requests.get(url, timeout=5).status_code == 200:
                print(f'서버 준비 완료 ({time.time()-t0:.0f}s)'); return True
        except Exception:
            # 연결 실패 시 무시하고 재시도
            pass
        # 5초마다 재시도
        time.sleep(5)
    # timeout 초과 시 예외 발생
    raise TimeoutError('서버가 안 떴습니다 — 터미널의 vllm serve 로그를 확인하세요.')

wait_ready()

서버 준비 완료 (0s)


True

**▸ 실행 결과 읽기.** `서버 준비 완료 (0s)` 는 `/v1/models` 가 즉시 200을 반환했다는 뜻 — 터미널 서버가 **이미 떠 있던** 상태라 0초 만에 통과했습니다. 서버를 막 띄운 직후라면 모델 로드 때문에 **수십 초~몇 분**이 찍힐 수 있습니다. 만약 `TimeoutError` 가 나면 서버가 안 뜬 것이니 **터미널의 `vllm serve` 로그**를 확인하세요.

## 4. OpenAI 클라이언트 + 벤치 함수

vLLM은 **OpenAI API와 호환**되므로 익숙한 `OpenAI(...)` 클라이언트로 그대로 호출합니다. 이미지는 **data-URI**(이미지를 base64 텍스트로 인코딩한 문자열)로 변환해 텍스트 질문과 함께 보냅니다.

두 성능 지표를 정의합니다 — 둘은 다릅니다:
- **지연(latency)**: 요청 1건을 보내고 답을 받기까지의 시간. 
- **처리량(throughput)**: 여러 요청을 **동시에** 보낼 때 초당 처리량(req/s). 

벤치는 결정적 재현을 위해 **`temperature=0`** 으로 고정합니다.

In [ ]:
# ── OpenAI 호환 클라이언트 + 벤치 함수 정의 ───────────────────────
import base64, time
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from datasets import load_dataset

# 로컬에서 실행 중인 vLLM(OpenAI 호환 API)에 연결
# - base_url: Day2-5에서 띄운 서버 주소
# - api_key: 로컬 서버라 임의값 사용 가능
# - timeout: VLM 추론이 길어질 수 있어 넉넉히 설정
client = OpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='EMPTY', timeout=600)
# vLLM은 Openai 기본으로 지원!!!  https://wikidocs.net/338594 읽어보자
# vLLM KV cache문제 : MHA => GQA   paged attention, continuouse batching

def to_data_uri(pil_image):
    """
    PIL 이미지를 data URI 문자열로 변환.
    vLLM chat/completions의 image_url 입력에 바로 사용한다.
    """
    # 메모리 버퍼에 PNG로 저장
    buf = BytesIO()
    pil_image.save(buf, format='PNG')
    # base64 인코딩 후 data URI 접두어와 결합
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def one_request(image, question, max_tokens=32):
    """
    이미지+질문 1건을 서버로 보내고 응답 객체를 반환.
    """
    return client.chat.completions.create(
        model=MODEL_NAME,  # 서버 실행 시 --served-model-name 과 동일해야 함
        messages=[{
            'role': 'user',
            'content': [
                # 멀티모달 입력: 이미지
                {'type': 'image_url', 'image_url': {'url': to_data_uri(image)}},
                # 멀티모달 입력: 텍스트 질문
                {'type': 'text', 'text': question}
            ]
        }],
        max_tokens=max_tokens,  # 출력 토큰 상한
        temperature=0           # 벤치 재현성을 위해 결정적 생성
    )

def measure_latency(samples):
    """
    순차 호출 평균 지연(초) 측정.

    - 샘플을 1개씩 직렬로 요청해, 각 요청의 왕복 시간(RTT)을 잰다.
    - 반환값은 `평균 latency(초)`이며, 단건 사용자 체감 속도에 가깝다.
    - 빈 입력이면 분모 0을 피하기 위해 ValueError를 발생시킨다.
    """
    if len(samples) == 0:
        raise ValueError('samples가 비어 있습니다.')

    lats = []
    for s in samples:
        t0 = time.time()
        one_request(s['image'], s['query'])  # 1건 요청(순차)
        lats.append(time.time() - t0)        # 요청별 소요 시간(초)

    return sum(lats) / len(lats)


def measure_throughput(samples, concurrency=16):
    """
    동시 호출 처리량(req/s) 측정.

    - 전체 샘플을 병렬로 보내고, 총 완료 시간으로 처리량을 계산한다.
    - 처리량 공식: `len(samples) / 총소요시간`.
    - 동시성은 최대 `concurrency`를 사용하되, 샘플 수보다 클 필요는 없어 자동으로 제한한다.
    - 빈 입력이면 ValueError를 발생시킨다.
    """
    if len(samples) == 0:
        raise ValueError('samples가 비어 있습니다.')

    workers = min(concurrency, len(samples))
    t0 = time.time()

    # 지정한 동시성으로 요청을 병렬 전송
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(lambda s: one_request(s['image'], s['query']), samples))

    dt = time.time() - t0
    return len(samples) / dt

# 벤치용 ChartQA 샘플 준비
# - bench_lat: 지연 측정용(8개, 순차)
# - bench_tp : 처리량 측정용(48개, 동시)
bench_lat = load_dataset('HuggingFaceM4/ChartQA', split='test').shuffle(seed=0).select(range(8))
bench_tp  = load_dataset('HuggingFaceM4/ChartQA', split='test').shuffle(seed=0).select(range(48))

print('벤치 샘플 준비: 지연 8개, 처리량 48개')

벤치 샘플 준비: 지연 8개, 처리량 48개


**▸ 실행 결과 읽기.** `벤치 샘플 준비: 지연 8개, 처리량 48개` — ChartQA test를 셔플해 지연용 8개·처리량용 48개를 골랐습니다(아직 측정 전, 함수만 정의됨). 적은 샘플이라 절대 수치보다 **원본↔AWQ의 상대 비교**용으로 보세요.

## 5. AWQ 8B 라이브 벤치 (단건 확인 + 지연·처리량·VRAM)

서버가 떠 있는 상태에서 ① 응답이 멀쩡한지 **단건으로 눈 확인**, ② **지연·처리량·VRAM**을 측정합니다.

> ⚠️ 여기서 재는 **서빙 VRAM은 모델 크기가 아닙니다.** `--gpu-memory-utilization 0.9` 때문에 vLLM이 KV 캐시용으로 GPU의 ~90%를 **미리 예약**하므로, 80GB A100이면 70GB 안팎으로 보입니다. **모델 자체 크기(≈6.7GB)** 는 6.절의 Day2-4 오프라인 수치로 확인합니다.

In [ ]:
# ── AWQ 8B 라이브 벤치 ──────────────────────────────────────
import subprocess

def gpu_mem_used_gb():
    # nvidia-smi로 현재 GPU 사용 메모리(MiB)를 조회하여 GB로 변환
    out = subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits',
                         shell=True, text=True, capture_output=True).stdout
    # 첫 번째 GPU의 값만 사용 (멀티 GPU 환경 대비)
    return float(out.strip().split('\n')[0]) / 1024

# 벤치 결과를 저장할 딕셔너리 
results = {}

# ── 단건 확인: 모델 응답이 정상인지 눈으로 검증 ──
ex = bench_lat[0]  # 지연 측정용 샘플 중 첫 번째 항목
r = one_request(ex['image'], ex['query'])  # 이미지+질문을 서버로 전송
print('Q:', ex['query'], '| 정답:', ex['label'][0], '| 모델:', r.choices[0].message.content)

# ── 라이브 서빙 지표 측정 ──
# VRAM: 측정 시점의 GPU 사용 메모리 (vLLM KV 캐시 예약분 포함)
vram = gpu_mem_used_gb()
# 지연: bench_lat(8개)를 순차 호출하여 평균 응답 시간(초) 측정
lat  = measure_latency(bench_lat)
# 처리량: bench_tp(48개)를 동시 호출하여 초당 처리 요청 수 측정
tp   = measure_throughput(bench_tp)

# 결과를 딕셔너리에 저장 (CELL INDEX 14의 비교 셀에서 사용)
results['8B AWQ'] = {'latency_s': lat, 'throughput_rps': tp, 'serving_vram_gb': vram}
print(f'8B AWQ(서빙) → 지연 {lat:.2f}s | 처리량 {tp:.2f} req/s | 서빙 VRAM {vram:.1f} GB(KV캐시 예약 포함)')

Q: What was the endowment fund value of Harvard University in dollars in 2020? | 정답: 40.58 | 모델: Based on the provided bar chart, we can determine the endowment fund value for Harvard University.

The chart is titled "Endowment fund value in billion U.S
8B AWQ(서빙) → 지연 0.85s | 처리량 9.99 req/s | 서빙 VRAM 70.7 GB(KV캐시 예약 포함)


**▸ 실행 결과 읽기.**

- **단건 응답**: 정답은 `40.58`인데 모델은 *"Based on the provided bar chart … Endowment fund value in billion U.S"* 처럼 **장황하게** 답했습니다. `max_tokens=32`로 잘린 데다, **간결한 숫자 답을 강제하는 system prompt가 없어서**입니다. 이 장황함은 아래 6.절의 정확도 결과로 이어집니다.
- **지연 0.85s**: 한 건을 순차로 보낼 때 평균 응답 시간. VLM은 이미지 인코딩(ViT) 비용이 있어 텍스트 전용 LLM보다 깁니다.
- **처리량 9.99 req/s**: 동시성 16으로 보냈을 때. 순차 환산(1 ÷ 0.85 ≈ 1.18 req/s)의 약 **8배** — vLLM의 **연속 배칭(continuous batching)** 이 여러 요청을 묶어 GPU를 꽉 채운 결과입니다.
- **서빙 VRAM 70.7GB**: 위에서 설명한 **KV 캐시 예약분**(0.9 × 80GB). 모델 크기가 아니라 "처리량을 위해 잡아둔 좌석"에 가깝습니다.

## 6. 비교 — AWQ 서빙(라이브) + 정확도·크기(Day2-4 오프라인)

원본 8B를 GPU에 **다시 띄우면 느리므로**, 정확도·크기는 Day2-4가 저장한 `eval_results.json` 에서 가져오고(오프라인), **서빙 속도/처리량만 AWQ 라이브**로 잰 값을 합쳐 봅니다. 이렇게 하면 시간 손실 없이 "양자화로 무엇이 좋아졌나"를 한눈에 비교할 수 있습니다.

In [ ]:
# ── 비교: AWQ 라이브 서빙 + 정확도/크기(Day2-4 오프라인) ───────────
import os, json  # 파일 존재 확인/경로 처리, JSON 로드·저장에 사용

# (1) Day2-4 오프라인 평가 결과(정확도·크기·로드VRAM) 로드
#     - 원본 8B를 다시 서빙하지 않고, 이전 교시 결과 파일로 비교
eval_path = f'{VLM_DIR}/eval_results.json'
offline = {}

if os.path.exists(eval_path):
    # eval_results.json이 있으면 로드
    with open(eval_path, encoding='utf-8') as f:
        offline = json.load(f)
else:
    # 파일이 없으면 경고만 출력하고, 라이브 서빙 지표만 표시
    print('⚠ eval_results.json 없음 — Day2-4(평가)를 먼저 실행하면 정확도·크기 비교가 채워집니다.\n')

# 직전 셀에서 측정한 AWQ 라이브 서빙 결과
a = results['8B AWQ']

print('── 서빙 성능 (AWQ, 라이브 측정) ──')
print(
    f'  지연 {a["latency_s"]:.2f}s | 처리량 {a["throughput_rps"]:.2f} req/s '
    f'| 서빙 VRAM {a["serving_vram_gb"]:.1f} GB(=vLLM KV캐시 예약 포함)'
)

# 오프라인 비교용: 원본 8B / AWQ 8B
o8, a8 = offline.get('8B 원본'), offline.get('8B AWQ')

# 두 항목이 모두 있을 때만 정확도·크기 비교 표 출력
if o8 and a8:
    print('\n── 정확도·크기 (Day2-4 오프라인) ──')
    print(f'{"모델":<10}{"Relaxed Acc":>12}{"크기(GB)":>11}{"로드VRAM(GB)":>13}')
    print('-' * 46)

    # 행 단위 출력: 모델명, 정확도, 디스크 크기, 로드 시 VRAM
    for nm, d in [('8B 원본', o8), ('8B AWQ', a8)]:
        print(f'{nm:<10}{d["relaxed_acc"]:>12.3f}{d["size_gb"]:>11.1f}{d["load_vram_gb"]:>13.1f}')

    print('-' * 46)

    # AWQ 압축률(크기 절감) 요약
    print(
        f'AWQ 크기 절감 : {o8["size_gb"]:.1f} → {a8["size_gb"]:.1f} GB '
        f'({(1 - a8["size_gb"]/o8["size_gb"])*100:.0f}% ↓)'
    )

    # AWQ 정확도 손실(원본 - AWQ), 절대값이 작을수록 바람직
    print(f'AWQ 정확도 손실: {o8["relaxed_acc"] - a8["relaxed_acc"]:+.3f} (작을수록 좋음)')

# (2) 서빙 벤치 저장
# - 라이브 서빙 결과는 항상 저장
# - 오프라인 결과가 있으면 함께 저장
serv = {'8B AWQ (serving)': a}
if offline:
    serv['offline_eval'] = offline

# JSON 파일로 저장(한글 유지, 보기 좋게 indent 적용)
with open(f'{VLM_DIR}/serving_bench.json', 'w', encoding='utf-8') as f:
    json.dump(serv, f, ensure_ascii=False, indent=2)

print('\n서빙 벤치 저장:', f'{VLM_DIR}/serving_bench.json')

── 서빙 성능 (AWQ, 라이브 측정) ──
  지연 0.85s | 처리량 9.99 req/s | 서빙 VRAM 70.7 GB(=vLLM KV캐시 예약 포함)

── 정확도·크기 (Day2-4 오프라인) ──
모델         Relaxed Acc     크기(GB)   로드VRAM(GB)
----------------------------------------------
8B 원본            0.000       16.3         16.3
8B AWQ           0.000        6.7          6.7
----------------------------------------------
AWQ 크기 절감 : 16.3 → 6.7 GB (59% ↓)
AWQ 정확도 손실: +0.000 (작을수록 좋음)

서빙 벤치 저장: /workspace/VLM_FT_2/serving_bench.json


**▸ 실행 결과 읽기.**

**핵심 성과 — 크기.** AWQ 4비트 양자화로 모델이 **16.3GB → 6.7GB (약 59% ↓)** 로 줄었고, 로드 VRAM도 같은 비율로 감소했습니다. 더 작은 GPU에 올리거나, 같은 GPU에서 더 많은 동시 요청을 받을 수 있다는 뜻 — 이게 이 교시의 목표입니다.

**서빙 VRAM(70.7) vs 로드 VRAM(6.7)** 은 측정 대상이 다릅니다. 전자는 vLLM이 **KV 캐시까지 예약**한 서빙 시점 점유량, 후자는 **모델 가중치만** 올렸을 때입니다. 모델이 "작다"는 근거는 후자(6.7GB)입니다.

**⚠️ 정확도 0.000 — 모델 탓이 아닙니다.** 원본과 AWQ **둘 다** Relaxed Accuracy가 0.000인데, 이는 "두 모델 다 못 푼다"가 아니라 **평가 포맷 문제**일 가능성이 큽니다. 5.절에서 봤듯 모델은 *"Based on the provided bar chart…"* 처럼 답하는데, Relaxed Accuracy 채점기는 응답에서 **숫자(예: 40.58)를 뽑아** 정답과 비교합니다 → 문장 속 숫자를 못 뽑으면 전부 오답 처리됩니다. 분모가 0이라 **양자화 손실(원본−AWQ)도 +0.000으로 무의미**합니다.

바로잡으려면 Day2-4 평가에서 ① **"숫자만 답하라"는 system prompt**를 주고, ② 응답에서 숫자를 추출하는 **파서를 보강**하면 됩니다. 그러면 두 모델 모두 정상 정확도가 나오고, AWQ의 진짜 손실(보통 1~2%p 수준)을 측정할 수 있습니다.

**정리.** 이 교시의 목적(서빙이 되고, 모델이 작아진다)은 달성됐습니다 — 59% 작아진 AWQ 모델이 OpenAI API로 정상 서빙되고 멀티모달 추론을 수행합니다. 다만 **정확도 비교는 Day2-4의 평가 포맷을 고친 뒤** 다시 봐야 의미가 있습니다. 결과는 `serving_bench.json` 에 저장됩니다.

## 7. 정리 + 다음 교시 예고

- **서버는 터미널에서 한 번** `vllm serve` 로 띄우고(블로킹·로그 실시간), 노트북은 **순수 클라이언트**입니다 — start/stop 반복이 없어 빠릅니다.
- **`--enforce-eager`** 로 기동 시간을 줄였습니다(데모/교육 트레이드오프).
- AWQ 서버에서 **지연·처리량**을 실측하고, **정확도·크기**는 Day2-4 오프라인 수치로 비교했습니다 — 원본 8B를 느리게 재기동하지 않습니다.

> ✅ **체크포인트**: ① `vllm serve` 가 OpenAI 호환 서버를 연다 ② data-URI로 이미지를 보낸다 ③ AWQ가 크기를 줄이면서 정확도 손실이 작다 ④ 서빙 VRAM은 KV 캐시 예약을 포함한다.

### 다음 교시 — Day2-6 · E2E 통합·정리
전체 파이프라인(QLoRA → 병합 → AWQ → vLLM 서빙)을 한 줄기로 되짚고, 산출물·수치를 정리합니다.